In [2]:
!pip install xgboost

   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/69.5 MB ? eta -:--:--
    --------------------------------------- 1.3/69.5 MB 4.9 MB/s eta 0:00:14
   - -------------------------------------- 2.4/69.5 MB 4.5 MB/s eta 0:00:15
   - -------------------------------------- 3.4/69.5 MB 4.7 MB/s eta 0:00:14
   -- ------------------------------------- 4.5/69.5 MB 4.6 MB/s eta 0:00:15
   --- ------------------------------------ 5.2/69.5 MB 4.5 MB/s eta 0:00:15
   --- ------------------------------------ 6.3/69.5 MB 4.5 MB/s eta 0:00:14
   ---- ----------------------------------- 7.1/69.5 MB 4.5 MB/s eta 0:00:15
   ---- ----------------------------------- 8.1/69.5 MB 4.5 MB/s eta 0:00:14
   ----- ---------------------------------- 9.4/69.5 MB 4.6 MB/s eta 0:00:14
   ----- ---------------------------------- 10.0/69.5 MB 4.4 MB/s eta 0:00:14
   ------ --------------------------------- 11.0/69.5 MB 4.5 MB/s eta 0:00:14
   ------ -

In [6]:
import pandas as pd
import numpy as np
import snowflake.connector
from cryptography.hazmat.primitives import serialization
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb
import joblib

pd.set_option('display.max_columns', None)

In [8]:
import pandas as pd
import snowflake.connector
from cryptography.hazmat.primitives import serialization

with open(r"C:\Users\reddy\OneDrive\Desktop\cat-fleet-predictive-maintenance\rsa_key.p8", "rb") as key_file:
    private_key = serialization.load_pem_private_key(key_file.read(), password=None)

pkb = private_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

# 2. Establish the Snowflake connection
sf_conn = snowflake.connector.connect(
    account='TGB56921',
    user='SPT',
    private_key=pkb,
    warehouse='SPT_WH',
    database='CAT_FLEET_DB',
    schema='PUBLIC'
)

# 3. Use the cursor with fetch_pandas_all() to avoid the pandas SQLAlchemy warning
cursor = sf_conn.cursor()
try:
    cursor.execute("SELECT * FROM GOLD_TELEMATICS")
    df_gold = cursor.fetch_pandas_all()
finally:
    cursor.close()

# 4. Close the connection
sf_conn.close()

print(f"Loaded {len(df_gold)} rows, {len(df_gold.columns)} columns from Snowflake")

Loaded 1970 rows, 56 columns from Snowflake


In [18]:
LEAKAGE_COLS = [
    'ENGINE_FAILURE_IMMINENT', 'BRAKE_ISSUE_IMMINENT', 'BATTERY_ISSUE_IMMINENT',
    'FAILURE_DATE', 'FAILURE_TYPE', 'CLEAN_FAILURE_TYPE',
    'FAILURE_TIMESTAMP', 'HOURS_UNTIL_FAILURE', 'TARGET_ENCODED'
]
ID_COLS = ['MACHINE_ID', 'TIMESTAMP', 'GPS_LATITUDE', 'GPS_LONGITUDE']
LABEL_COL = 'FINAL_TARGET_LABEL'
DROP_COLS = LEAKAGE_COLS + ID_COLS + [LABEL_COL, 'BRAND']

id_reference_df = df_gold[ID_COLS + [LABEL_COL]].copy()
df_gold['BINARY_LABEL'] = (df_gold[LABEL_COL] != 'Healthy').astype(int)

y = df_gold['BINARY_LABEL']
X = df_gold.drop(columns=DROP_COLS + ['BINARY_LABEL'])

print(f"Feature matrix shape: {X.shape}")
print(f"Label distribution:\n{y.value_counts()}")
print(f"\nMissing values per column:\n{X.isnull().sum()[X.isnull().sum() > 0]}")

Feature matrix shape: (1970, 41)
Label distribution:
BINARY_LABEL
0    1945
1      25
Name: count, dtype: int64

Missing values per column:
Series([], dtype: int64)


In [19]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = ['precision', 'recall', 'f1', 'roc_auc']

# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=8, class_weight='balanced',
    random_state=42, n_jobs=-1
)
rf_cv = cross_validate(rf_model, X, y, cv=skf, scoring=scoring)

# XGBoost
scale_pos_weight = (y == 0).sum() / (y == 1).sum()
xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=42
)
xgb_cv = cross_validate(xgb_model, X, y, cv=skf, scoring=scoring)

print("=== Random Forest (5-fold CV) ===")
for metric in scoring:
    scores = rf_cv[f'test_{metric}']
    print(f"{metric:>10}: {scores.mean():.3f} ± {scores.std():.3f}  | folds: {np.round(scores, 2)}")

print("\n=== XGBoost (5-fold CV) ===")
for metric in scoring:
    scores = xgb_cv[f'test_{metric}']
    print(f"{metric:>10}: {scores.mean():.3f} ± {scores.std():.3f}  | folds: {np.round(scores, 2)}")

C:\Users\reddy\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\reddy\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\reddy\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\reddy\anaconda3\Lib\site-packages\sklearn\metrics\_clas

=== Random Forest (5-fold CV) ===
 precision: 0.000 ± 0.000  | folds: [0. 0. 0. 0. 0.]
    recall: 0.000 ± 0.000  | folds: [0. 0. 0. 0. 0.]
        f1: 0.000 ± 0.000  | folds: [0. 0. 0. 0. 0.]
   roc_auc: 0.985 ± 0.023  | folds: [0.94 1.   1.   1.   1.  ]

=== XGBoost (5-fold CV) ===
 precision: 0.691 ± 0.185  | folds: [1.   0.6  0.5  0.56 0.8 ]
    recall: 0.720 ± 0.160  | folds: [0.6 0.6 0.6 1.  0.8]
        f1: 0.682 ± 0.095  | folds: [0.75 0.6  0.55 0.71 0.8 ]
   roc_auc: 0.996 ± 0.002  | folds: [1.   0.99 1.   1.   1.  ]


In [22]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=42
)
xgb_model.fit(X_train, y_train)

holdout_preds = xgb_model.predict(X_test)

print("=== Single Holdout Split ===")
print(classification_report(y_test, holdout_preds, target_names=['Healthy', 'Failure']))
print(f"Confusion Matrix:\n{confusion_matrix(y_test, holdout_preds)}")

=== Single Holdout Split ===
              precision    recall  f1-score   support

     Healthy       0.99      1.00      1.00       389
     Failure       0.75      0.60      0.67         5

    accuracy                           0.99       394
   macro avg       0.87      0.80      0.83       394
weighted avg       0.99      0.99      0.99       394

Confusion Matrix:
[[388   1]
 [  2   3]]


In [26]:
importance_df=pd.DataFrame({
    'features':X.columns,
    'importance':xgb_model.feature_importances_
}).sort_values('importance',ascending=False)
print("top 15 most important featurees :")
print(importance_df.head(15).to_string(index=False))

top 15 most important featurees :
                        features  importance
           BRAKE_FLUID_LEVEL_PSI    0.257160
               BATTERY_VOLTAGE_V    0.247731
                 VIBRATION_LEVEL    0.242816
      VIBRATION_LEVEL_ROLL3_MEAN    0.075353
    BATTERY_VOLTAGE_V_ROLL3_MEAN    0.044354
         BRAKE_PEDAL_POS_PERCENT    0.017514
                    BRAKE_TEMP_C    0.013107
     OIL_PRESSURE_PSI_ROLL3_MEAN    0.011300
BRAKE_FLUID_LEVEL_PSI_ROLL3_MEAN    0.010685
         BRAKE_TEMP_C_ROLL3_MEAN    0.009011
               BATTERY_CURRENT_A    0.008037
              WHEEL_SPEED_RL_KPH    0.007960
                  AMBIENT_TEMP_C    0.005817
                OIL_PRESSURE_PSI    0.005562
                  BATTERY_TEMP_C    0.004281


In [28]:
import joblib

scale_pos_weight_full = (y == 0).sum() / (y == 1).sum()

final_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight_full, eval_metric='logloss', random_state=42
)
final_model.fit(X, y)

joblib.dump(final_model, 'cat_fleet_failure_model.pkl')
print("Final model trained on full dataset (1970 rows) and saved: cat_fleet_failure_model.pkl")

joblib.dump(X.columns.tolist(), 'model_features.pkl')
print("Feature column list saved: model_features.pkl")

Final model trained on full dataset (1970 rows) and saved: cat_fleet_failure_model.pkl
Feature column list saved: model_features.pkl


In [29]:
print("=" * 60)
print("MODEL TRAINING SUMMARY")
print("=" * 60)
print(f"Algorithm: XGBoost (chosen over Random Forest)")
print(f"Reason: RF predicted 0% recall on failures at default threshold;")
print(f"        XGBoost achieved ~72% recall via scale_pos_weight")
print(f"Training data: {len(df_gold)} rows, {X.shape[1]} features")
print(f"Class distribution: {(y==0).sum()} Healthy, {(y==1).sum()} Failure")
print(f"Cross-validated performance (5-fold):")
print(f"  Recall:    0.720 ± 0.160")
print(f"  Precision: 0.691 ± 0.185")
print(f"  F1:        0.682 ± 0.095")
print(f"  ROC-AUC:   0.996 ± 0.002")
print("=" * 60)

MODEL TRAINING SUMMARY
Algorithm: XGBoost (chosen over Random Forest)
Reason: RF predicted 0% recall on failures at default threshold;
        XGBoost achieved ~72% recall via scale_pos_weight
Training data: 1970 rows, 41 features
Class distribution: 1945 Healthy, 25 Failure
Cross-validated performance (5-fold):
  Recall:    0.720 ± 0.160
  Precision: 0.691 ± 0.185
  F1:        0.682 ± 0.095
  ROC-AUC:   0.996 ± 0.002
